# Analisis del Impacto Ambiental del Monocultivo de Aguacate en Michoacan
## HackODS UNAM 2026 | ODS 15: Vida de Ecosistemas Terrestres

Este estudio utiliza la estructura narrativa de la **Piramide de Freytag** para exponer la realidad ambiental de Michoacan, cruzando datos reales del Censo Agropecuario 2022, la Agenda 2030 y estadisticas forestales oficiales.

### Estructura de la Historia:
1. **Exposicion**: El auge economico y la dominancia nacional del aguacate.
2. **Conflicto**: La presion sobre el suelo forestal y el cambio de uso de suelo.
3. **Climax**: La degradacion del ecosistema y la perdida de biodiversidad (Pino/Encino).
4. **Resolucion**: El analisis de la brecha de apoyo y gestion ambiental.

In [5]:
import pandas as pd
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import plotly.express as px
import plotly.graph_objects as go

'''
AUDITORIA DE INTEGRIDAD DE DATOS
Validacion estricta de archivos procesados. Sin datos simulados.
'''

DATA_DIR = "../data/procesados/"
REQUIRED_FILES = {
    "dataset_maestro_ods15.csv": "Base de datos municipal de Michoacan.",
    "contexto_nacional_ods15.csv": "Comparativa interestatal nacional.",
    "biodiversidad_especies_ods15.csv": "Volumen maderable por especie.",
    "gestion_ambiental_ods15.csv": "Brecha de apoyo ambiental.",
    "impact_coefficients.json": "Coeficientes de impacto tecnico."
}

COLORS = {
    'verde': '#2E8B57',
    'cafe': '#8B4513',
    'miel': '#DAA520',
    'rojo': '#F44336'
}

print("INFO: Iniciando auditoria tecnica de datos...\n" + "="*50)

def audit_workspace(data_dir, files_dict):
    for filename, description in files_dict.items():
        path = os.path.join(data_dir, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(f"ERROR: El archivo {filename} no existe en {data_dir}.")
        
        print(f"VALIDO: {filename}")
        print(f"   Proposito: {description}")
        print("-"*50)

audit_workspace(DATA_DIR, REQUIRED_FILES)
print("EXITO: Entorno validado para visualizacion real.")

INFO: Iniciando auditoria tecnica de datos...
VALIDO: dataset_maestro_ods15.csv
   Proposito: Base de datos municipal de Michoacan.
--------------------------------------------------
VALIDO: contexto_nacional_ods15.csv
   Proposito: Comparativa interestatal nacional.
--------------------------------------------------
VALIDO: biodiversidad_especies_ods15.csv
   Proposito: Volumen maderable por especie.
--------------------------------------------------
VALIDO: gestion_ambiental_ods15.csv
   Proposito: Brecha de apoyo ambiental.
--------------------------------------------------
VALIDO: impact_coefficients.json
   Proposito: Coeficientes de impacto tecnico.
--------------------------------------------------
EXITO: Entorno validado para visualizacion real.


# 1. Exposicion: Crecimiento Vertical y Perdida de Masa Forestal
Analizamos la evolucion historica del aguacate frente a la perdida de cobertura arborea (datos de Global Forest Watch).

In [6]:
# Carga de datos historicos de perdida de cobertura (Real GFW 2001-2024)
path_loss = "data/crudos/globalforest/Tree cover loss in Michoacán, México/treecover_loss__ha.csv"
df_loss = pd.read_csv(path_loss)
df_loss = df_loss[df_loss['umd_tree_cover_loss__year'] >= 2012]

# Datos de produccion (Puntos reales 2017, 2019, 2022 interpolados para la tendencia)
# Nota: Se usan valores reales del Censo y SIAP para Michoacan
years = np.arange(2012, 2025)
prod_vals = {2012: 1.1, 2017: 1.9, 2019: 2.1, 2022: 3.42}
production = np.interp(years, list(prod_vals.keys()), list(prod_vals.values()))

fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.set_xlabel('Anio')
ax1.set_ylabel('Produccion Aguacate (Millones Ton)', color=COLORS['verde'])
ax1.plot(years, production, color=COLORS['verde'], marker='o', linewidth=2, label='Produccion (SIAP/INEGI)')
ax1.tick_params(axis='y', labelcolor=COLORS['verde'])

ax2 = ax1.twinx()
ax2.set_ylabel('Perdida de Cobertura Arborea (Ha)', color=COLORS['cafe'])
ax2.plot(df_loss['umd_tree_cover_loss__year'], df_loss['umd_tree_cover_loss__ha'], 
         color=COLORS['cafe'], linestyle='--', marker='x', alpha=0.7, label='Perdida Forestal (GFW)')
ax2.tick_params(axis='y', labelcolor=COLORS['cafe'])

plt.title('Crecimiento Vertical: Produccion vs Perdida de Cobertura (2012-2024)', pad=20)
fig.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'data/crudos/globalforest/Tree cover loss in Michoacán, México/treecover_loss__ha.csv'

### ¿Que es la Correlacion de Pearson?
El Coeficiente de Correlacion de Pearson (`r`) es una medida estadistica que cuantifica la fuerza y la direccion de la relacion lineal entre dos variables. 

- **Valor cercano a 1**: Indica una relacion positiva fuerte (si una aumenta, la otra tambien).
- **Valor cercano a 0**: Indica que no hay relacion lineal.
- **Valor cercano a -1**: Indica una relacion negativa fuerte.

En nuestro analisis, buscamos demostrar que el aumento en la superficie de aguacate esta **directamente vinculado** a una mayor presion forestal en los municipios.

In [ ]:
# 2. Correlacion Municipal Real (Censo 2022)
df_final = pd.read_csv(os.path.join(DATA_DIR, "dataset_maestro_ods15.csv"))

plt.figure(figsize=(10, 6))
sns.regplot(data=df_final, x='ha_aguacate', y='impact_ratio', 
            scatter_kws={'alpha':0.6, 'color': COLORS['verde']}, 
            line_kws={'color': COLORS['cafe'], 'linewidth': 2})

# Calculo de Pearson
corr, p_value = pearsonr(df_final['ha_aguacate'], df_final['impact_ratio'])

plt.title(f'Correlacion de Pearson: {corr:.2f}\nMunicipios de Michoacan (Censo 2022)', pad=15)
plt.xlabel('Superficie de Aguacate (Ha)')
plt.ylabel('Indice de Presion Forestal (Impact Ratio)')
plt.grid(True, linestyle='--', alpha=0.5)
sns.despine()
plt.show()

print(f"Interpretacion: Existe una correlacion de {corr:.2f}, lo que sugiere un vinculo {'fuerte' if corr > 0.7 else 'moderado'} entre la siembra y la presion forestal detectada.")